# Prompt Chaining

**Prompt Chaining** เป็นอีกหนึ่งเทคนิคในการใช้ LLMs ในงานที่ซับซ้อนขึ้น โดยเริ่มจากการย่อยงานที่ซับซ้อนให้กลายเป็นงานย่อยๆ ที่จัดการได้ง่ายขึ้น จากนั้นนำผลลัพธ์ (Output) จากคำสั่ง (Prompt) หนึ่งของ LLM มาใช้เป็นข้อมูลนำเข้า (Input) สำหรับคำสั่งถัดไปตามลำดับไปเรื่อยๆ จนได้ผลลัพท์ที่ต้องการ

ผลที่ได้ คือ 

* ช่วยให้มีแนวทางการทำงานที่เป็นระบบ มากยิ่งขึ้น

* เป็นการนำทาง LLM ผ่านลำดับขั้นตอนของการใช้เหตุผล

* นำไปสู่คำตอบที่แม่นยำ และครอบคลุมมากกว่าเดิม

* เพิ่มความโปร่งใส ในกระบวนการคิดของ AI


![Prompt Chaining](https://www.ibm.com/content/dam/connectedassets-adobe-cms/worldwide-content/creative-assets/s-migr/ul/g/26/b0/prompt-chaining-example.png "Prompt Chaining")

credit: https://www.ibm.com/think/topics/prompt-chaining



# Task

คุณคือ AI รับออเดอร์ร้านอาหารตามสั่งชื่อ "ร้านตามใจสั่ง" แห่งหนึ่ง งานของคุณ คือ ช่วยรับออเดอร์จากลูกค้า จากนั้นตรวจสอบความครบถ้วนของออเดอร์ จากนั้นก็คำนวนยอด โดยมีรายละเอียดดังนี้

## **Menu & Pricing**:

ข้าวไข่เจียว = 40 บาท (ไม่ต้องเลือกเนื้อสัตว์)

ข้าวกะเพรา = หมู/ไก่ 50 บาท, กุ้ง 55 บาท

ข้าวผัด = หมู/ไก่ 50 บาท, กุ้ง 55 บาท

ข้าวทอดกระเทียม = หมู/ไก่ 50 บาท, กุ้ง 55 บาท

ออปชั่นเสริม: ไข่ดาว = เพิ่ม 5 บาท (ใช้ได้กับทุกเมนู ยกเว้นข้าวไข่เจียว)

## **Workflow & Rules (ทำตามทีละขั้นตอน)**:

#### Step 1: การทักทาย
ให้เริ่มต้นบทสนทนาด้วยข้อความประมาณนี้:

```
สวัสดีคับ วันนี้รับอะไรดี? ที่ร้านมีอยู่ 4 เมนู คือ

* ข้าวไข่เจียว

* ข้าวกะเพรา (หมู, ไก่, กุ้ง)

* ข้าวผัด (หมู, ไก่, กุ้ง)

* ข้าวทอดกระเทียม (หมู, ไก่, กุ้ง)
```

#### Step 2: การเก็บข้อมูล
ทุกครั้งที่ลูกค้าตอบ ให้คุณวิเคราะห์เงื่อนไขต่อไปนี้:

* ลูกค้าสั่งเมนูอะไร?

* ถ้าไม่ใช่ข้าวไข่เจียว ลูกค้าระบุเนื้อสัตว์ (หมู, ไก่, หรือ กุ้ง) หรือยัง? (ต้องเลือกแค่อย่างเดียว)

* ลูกค้าระบุหรือยังว่าจะรับ "ไข่ดาว" ด้วยหรือไม่? (รับ หรือ ไม่รับ)

สำคัญ: หากข้อมูลยังไม่ครบ ห้ามคิดเงินเด็ดขาด ให้ถามกลับเฉพาะข้อมูลที่ยังขาดอยู่เท่านั้น เช่น "รับเป็นเนื้อสัตว์อะไรดีครับ (หมู/ไก่/กุ้ง)?" หรือ "รับไข่ดาวเพิ่มด้วยไหมครับ?"

#### Step 3: สรุปออเดอร์และคิดเงิน
เมื่อได้ข้อมูลครบถ้วนแล้ว ให้สรุปรายการอาหาร และแสดงการคำนวณราคาให้ลูกค้าดูอย่างชัดเจนตามฐานราคาที่กำหนดไว้


## STEP1: Design

In [2]:
from IPython.display import Image
from IPython.core.display import HTML 
Image(url= "./Examples/prompt-chain.png")

In [1]:
# !uv pip install langchain-ollama langgraph pydantic

## STEP2: Build Graph

In [3]:
from typing import TypedDict, Annotated, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    menu: Optional[str]
    meat: Optional[str]
    egg: Optional[str]

class Order(BaseModel):
    menu: Optional[str] = Field(description="เมนูที่ลูกค้าเลือก: 'ข้าวไข่เจียว', 'ข้าวกะเพรา', 'ข้าวผัด', 'ข้าวทอดกระเทียม' หรือ None if it does not specify")
    meat: Optional[str] = Field(description="เนื้อสัตว์ที่ลูกค้าเลือก: 'หมู', 'ไก่', 'กุ้ง' หรือ None if it does not specify")
    egg: Optional[str] = Field(description="ไข่ดาว: 'รับ' หรือ 'ไม่รับ' หรือ None if it does not specify")

# เตรียม LLM (ใช้ตัวที่รองรับ structured output)
llm = ChatOllama(model="scb10x/typhoon2.5-qwen3-4b", temperature=0)
order_extractor_llm = llm.with_structured_output(Order)

In [6]:
# order_extractor_llm.invoke([
#     SystemMessage(content="คุณคือผู้ช่วยร้านอาหารตามสั่ง หน้าที่ของคุณคือสกัดข้อมูลเมนูอาหาร เนื้อสัตว์ และไข่ดาว จากบทสนทนา"),
#     HumanMessage(content="อยากกินกระเพราอะ"),
# ])

### Define Nodes

In [59]:
def InfoExtractorNode(state: AgentState):
    """
    Node 1: อ่านประวัติแชททั้งหมด แล้วสกัดข้อมูลว่า ณ ปัจจุบันรู้อะไรแล้วบ้าง
    """

    messages = state["messages"]
    sys_prompt = "คุณคือผู้ช่วยร้านอาหารตามสั่ง หน้าที่ของคุณคือสกัดข้อมูลเมนูอาหาร เนื้อสัตว์ และไข่ดาว จากบทสนทนา"
    
    userMessages = []
    for m in messages:
        if type(m)==HumanMessage:
            userMessages += [m]
            
    result: Order = order_extractor_llm.invoke([SystemMessage(content=sys_prompt)] + userMessages)
    
    # อัปเดต State ล่าสุด
    return {"menu": result.menu, "meat": result.meat, "egg": result.egg}

In [60]:
def MissingInfoDetectorNode(state: AgentState):
    """
    Node 2: ถ้าข้อมูลไม่ครบ ให้สร้างคำถามถามลูกค้า (Human-in-the-loop Component)
    """
    missing = []
    if not state.get("menu"):
        missing.append("เมนูอาหาร")
    else:
        # ข้าวไข่เจียวไม่ต้องเลือกเนื้อสัตว์
        if state["menu"] != "ข้าวไข่เจียว" and not state.get("meat"):
            missing.append("เนื้อสัตว์ (หมู/ไก่/กุ้ง)")
    
    if not state.get("egg"):
        missing.append("รับไข่ดาวด้วยไหม")
        
    prompt = f"""
    จากข้อมูลปัจจุบัน: เมนู={state.get('menu')}, เนื้อ={state.get('meat')}, ไข่ดาว={state.get('egg')}
    ข้อมูลที่ยังขาดคือ: {', '.join(missing)}
    จงเขียนคำถามสั้นๆ สุภาพ เป็นกันเอง เพื่อถามเฉพาะสิ่งที่ยังขาดอยู่จากลูกค้า
    """
    response = llm.invoke(prompt)
    return {"messages": [response]}

In [61]:
def PriceCalculatorNode(state: AgentState):
    """
    Node 3: คำนวณราคาเมื่อข้อมูลครบ
    """
    price = 0
    menu = state.get("menu")
    meat = state.get("meat")
    egg = state.get("egg")
    
    # 1. คิดราคาหลัก
    if menu == "ข้าวไข่เจียว":
        price = 40
    else:
        if meat in ["หมู", "ไก่"]:
            price = 50
        elif meat == "กุ้ง":
            price = 55
            
    # 2. บวกค่าไข่ดาว
    if egg == "รับ":
        price += 5
        
    # 3. สร้างข้อความสรุป
    meat_text = f" {meat}" if meat else ""
    egg_text = " + ไข่ดาว" if egg == "รับ" else " (ไม่ใส่ไข่ดาว)"
    
    summary = f"สรุปออเดอร์ของคุณคือ: {menu}{meat_text}{egg_text}\nราคาทั้งหมด: {price} บาทครับ\nขอบคุณที่อุดหนุน 'ร้านตามใจสั่ง' ครับ!"
    
    return {"messages": [AIMessage(content=summary)]}

### Define Edges & Logic

In [62]:
def check_completeness_edge(state: AgentState) -> str:
    """
    Routing Logic: ตัววิเคราะห์ว่าควรไปถามต่อ (Ask) หรือไปคิดเงิน (Calculate)
    """
    menu = state.get("menu")
    meat = state.get("meat")
    egg = state.get("egg")
    
    if not menu:
        return "incomplete"
    
    if menu != "ข้าวไข่เจียว" and not meat:
        return "incomplete"
        
    if not egg:
        return "incomplete"
        
    return "complete"

### Define the graph

In [63]:
workflow = StateGraph(AgentState)

### Try

In [ ]:
initial_greeting = "รับอะไรดีครับ ที่ร้านมีอยู่ 4 เมนู คือ ข้าวไข่เจียว, ข้าวกะเพรา (หมู, ไก่, กุ้ง), ข้าวผัด (หมู, ไก่, กุ้ง) หรือ ข้าวทอดกระเทียม (หมู, ไก่, กุ้ง)"
print(f"Bot: {initial_greeting}")

# สร้าง State เริ่มต้น
current_state = {"messages": [AIMessage(content=initial_greeting)]}

# วนลูปการสนทนา (Human in the loop)
while True:
    user_input = input("You: ")
    if user_input.lower() =="q":
        break
        
    current_state["messages"].append(HumanMessage(content=user_input))
    result = app.invoke(current_state)
    current_state = result
    
    # ปริ้นท์คำตอบล่าสุดของ Bot (คำถามถามต่อ หรือ สรุปราคา)
    print(f"Bot: {current_state['messages'][-1].content}")
    
    # ถ้า State มีการคำนวณราคา แปลว่าจบออเดอร์แล้ว ให้ออกจากลูป
    if "ราคาทั้งหมด" in current_state['messages'][-1].content:
        break